In [1]:
import os, glob, json, shutil, logging, warnings
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
from rasterio.warp import reproject, Resampling

os.chdir(os.path.expanduser("~/projects/iride_onboard-burnscar-mapper"))
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("tiling")

# ---- shared class legend ----
MASK_CLEAR, MASK_FRESH, MASK_OLD = 0, 2, 3
MASK_CLOUD, MASK_SHADOW, MASK_WATER, MASK_NODATA = 4, 5, 6, 255
BAND_NAMES = ["B", "G", "R", "RE1", "RE2", "RE3", "NIR"]   # index 2=R, 6=NIR for NDVI
QUANTIFICATION = {"phisat2_sim": 10000.0, "heo": 4094.0}

# ---- paths ----
PHI_DIR, HEO_DIR = "data/PHI", "data/HEO"
OUT_ROOT = "processed/dataset_v1"
TILE = 256

# ---- curation ----
MAX_NODATA_INFORMATIVE = 0.5    # burn/cloud/shadow/water tiles are precious
MAX_NODATA_CLEAR = 0.05         # clear tiles are abundant - only keep near-complete ones
NDVI_HARD_NEG_THRESHOLD = 0.25  # set from real NDVI-by-class distributions (see diagnostic cell)
CLEAR_KEEP_RATIO_EASY = 0.10    # ordinary background clear tiles (NDVI >= threshold)
CLEAR_KEEP_RATIO_HARD = 1.00    # confusable clear tiles (NDVI < threshold) - keep essentially all

# ---- test-time multi-GSD resampling ----
ENABLE_MULTI_GSD_TEST = True
TEST_GSDS = {"native": 4.75, "heo_like": 2.5, "s2_like": 10.0}

# ---- safety ----
MIN_FREE_GB = 15

# confirmed via 03_Source_Stats: OCM under-detected snow/haze contamination
# (RE2 p99 > 1.0, physically implausible for clear terrain) - masks unreliable
HEO_EXCLUDE = {
    "IMH04_1CST__OPT8_20260317T123240_20260317T123242_20260323T105659_03985______O_A02",
    "IMH04_1CST__OPT8_20260317T123336_20260317T123338_20260319T102551_03985______O_A02",
    "IMH04_1CST__OPT8_20260317T123338_20260317T123339_20260319T102242_03985______O_A02",
    "IMH04_1CST__OPT8_20260317T123343_20260317T123345_20260323T115009_03985______O_A02",
    "IMH04_1CST__OPT8_20260317T123341_20260317T123343_20260323T114815_03985______O_A02",
    "IMH09_1CST__OPT8_20260722T124549_20260722T124552_20260722T155456_05875______O_A02",  # confirmed corrupted this session: per-band noise (not misregistration -- each band individually incoherent), absent from clean-scene comparison
}

def free_gb(path="."):
    return shutil.disk_usage(path).free / 1e9

print(f"free disk: {free_gb():.1f} GB")

free disk: 117.5 GB


In [2]:
def split_scenes(scenes, frac_train, frac_val, seed=42):
    scenes = list(scenes)
    rng = np.random.default_rng(seed)
    rng.shuffle(scenes)
    n = len(scenes)
    if n < 3:
        return {s: "train" for s in scenes}
    n_train = min(max(1, round(n * frac_train)), n - 2)
    n_val = min(max(1, round(n * frac_val)), n - n_train - 1)
    return {**{s: "train" for s in scenes[:n_train]},
           **{s: "val" for s in scenes[n_train:n_train+n_val]},
           **{s: "test" for s in scenes[n_train+n_val:]}}

phi_scenes = sorted(os.path.basename(p)[:-8] for p in glob.glob(f"{PHI_DIR}/images/*_img.tif"))
phi_map = split_scenes(phi_scenes, frac_train=0.8, frac_val=0.1)

heo_scenes_all = sorted(os.path.basename(p)[:-8] for p in glob.glob(f"{HEO_DIR}/images/*_img.tif"))
heo_scenes = [s for s in heo_scenes_all if s not in HEO_EXCLUDE]
heo_map_raw = split_scenes(heo_scenes, frac_train=0.7, frac_val=0.15)
heo_map = {s: f"heo_{v}" for s, v in heo_map_raw.items()}

print(f"PhiSat-2: {len(phi_scenes)} scenes -> {pd.Series(phi_map).value_counts().to_dict()}")
print(f"HEO: {len(heo_scenes)} usable ({len(heo_scenes_all)-len(heo_scenes)} excluded) "
     f"-> {pd.Series(heo_map).value_counts().to_dict()}")

PhiSat-2: 383 scenes -> {'train': 306, 'test': 39, 'val': 38}
HEO: 20 usable (5 excluded) -> {'heo_train': 14, 'heo_val': 3, 'heo_test': 3}


In [3]:
def resample_scene(img, mask, src_res, target_res, src_transform, crs):
    """Image: bilinear. Mask: NEAREST always - never invent intermediate classes."""
    if target_res == src_res:
        return img, mask, src_transform
    scale = src_res / target_res
    H, W = mask.shape
    nH, nW = max(1, round(H * scale)), max(1, round(W * scale))
    new_t = rasterio.transform.from_origin(src_transform.c, src_transform.f, target_res, target_res)
    img_out = np.zeros((img.shape[0], nH, nW), img.dtype)
    for b in range(img.shape[0]):
        reproject(img[b], img_out[b], src_transform=src_transform, src_crs=crs,
                  dst_transform=new_t, dst_crs=crs, resampling=Resampling.bilinear)
    mask_out = np.zeros((nH, nW), mask.dtype)
    reproject(mask, mask_out, src_transform=src_transform, src_crs=crs,
              dst_transform=new_t, dst_crs=crs, resampling=Resampling.nearest)
    return img_out, mask_out, new_t

def compute_ndvi(img, scale):
    """img: (7,H,W) raw DN, band order B,G,R,RE1,RE2,RE3,NIR -> R=idx2, NIR=idx6."""
    red = img[2].astype(np.float32) / scale
    nir = img[6].astype(np.float32) / scale
    return (nir - red) / np.maximum(nir + red, 1e-6)

def curate_and_tile(img, mask, scene_id, source, split, gsd_tag, scale,
                    out_root=OUT_ROOT, tile=TILE):
    """Hard-negative-aware curation:
    - informative tiles (any burn, or >=5% cloud/shadow/water): always kept,
      loose nodata tolerance since these are rare and valuable.
    - clear tiles: strict nodata tolerance (abundant, keep only clean ones).
      NDVI computed per-tile over clear pixels; low-NDVI ("hard negative" -
      bare soil/rock/sparse scrub that spectrally resembles burn scars) are
      kept at ~100%; ordinary high-NDVI clear tiles thinned to 10%.
    """
    rng = np.random.default_rng(abs(hash((scene_id, gsd_tag))) % (2**32))
    sub = f"{out_root}/{split}"
    os.makedirs(sub, exist_ok=True)
    C, H, W = img.shape
    ndvi = compute_ndvi(img, scale)
    rows = []
    for y in range(0, H - tile + 1, tile):
        for x in range(0, W - tile + 1, tile):
            m = mask[y:y+tile, x:x+tile]
            nodata_frac = (m == MASK_NODATA).mean()
            fr = {c: float((m == c).mean()) for c in
                  [MASK_CLEAR, MASK_FRESH, MASK_OLD, MASK_CLOUD, MASK_SHADOW, MASK_WATER]}
            informative = (fr[MASK_FRESH] > 0 or fr[MASK_OLD] > 0 or
                          fr[MASK_CLOUD] >= 0.05 or fr[MASK_SHADOW] >= 0.05 or fr[MASK_WATER] >= 0.05)

            hard_neg = False
            if informative:
                if nodata_frac > MAX_NODATA_INFORMATIVE:
                    continue
                keep = True
            else:
                if nodata_frac > MAX_NODATA_CLEAR:
                    continue
                clear_px = ndvi[y:y+tile, x:x+tile][m == MASK_CLEAR]
                tile_ndvi = float(np.nanmean(clear_px)) if clear_px.size else np.nan
                hard_neg = bool(not np.isnan(tile_ndvi) and tile_ndvi < NDVI_HARD_NEG_THRESHOLD)
                keep_ratio = CLEAR_KEEP_RATIO_HARD if hard_neg else CLEAR_KEEP_RATIO_EASY
                keep = rng.random() <= keep_ratio

            if not keep:
                continue
            base = f"{sub}/{scene_id}_{gsd_tag}_{y:05d}_{x:05d}"
            if os.path.exists(base + "_img.npy"):   # resume-safe
                continue
            np.save(base + "_img.npy", img[:, y:y+tile, x:x+tile])
            np.save(base + "_mask.npy", m)
            rows.append(dict(tile=os.path.basename(base), scene=scene_id, source=source,
                             split=split, gsd=gsd_tag, scale=scale, informative=informative,
                             hard_negative=hard_neg, nodata_frac=round(nodata_frac, 4),
                             **{f"frac_{k}": round(v, 4) for k, v in fr.items()}))
    return rows

In [4]:
RUN_NDVI_DIAGNOSTIC = False   # set True to regenerate the plot/stats

if RUN_NDVI_DIAGNOSTIC:
    ndvi_by_class = {c: [] for c in [MASK_CLEAR, MASK_FRESH, MASK_OLD]}
    for scene in phi_scenes[:40]:
        with rasterio.open(f"{PHI_DIR}/images/{scene}_img.tif") as s: img = s.read()
        with rasterio.open(f"{PHI_DIR}/masks/{scene}_mask.tif") as s: msk = s.read(1)
        ndvi = compute_ndvi(img, QUANTIFICATION["phisat2_sim"])
        for c in ndvi_by_class:
            px = ndvi[msk == c]
            if px.size:
                ndvi_by_class[c].append(px[np.random.choice(px.size, min(50000, px.size), replace=False)])

    for c, name in [(MASK_CLEAR,"clear"), (MASK_FRESH,"fresh_burn"), (MASK_OLD,"old_burn")]:
        vals = np.concatenate(ndvi_by_class[c]) if ndvi_by_class[c] else np.array([])
        if vals.size:
            print(f"{name}: mean={vals.mean():.3f} p10={np.percentile(vals,10):.3f} "
                 f"p50={np.percentile(vals,50):.3f} p90={np.percentile(vals,90):.3f}")

    fig, ax = plt.subplots(figsize=(8, 4))
    for c, name, color in [(MASK_CLEAR,"clear","tab:green"), (MASK_FRESH,"fresh","tab:red"),
                           (MASK_OLD,"old","tab:brown")]:
        vals = np.concatenate(ndvi_by_class[c]) if ndvi_by_class[c] else np.array([])
        if vals.size: ax.hist(vals, bins=80, range=(-0.5,1), alpha=0.5, density=True, label=name, color=color)
    ax.set_xlabel("NDVI"); ax.legend(); ax.set_title("NDVI distribution by class")
    plt.show()
# Result from the actual run (382-scene PhiSat-2 dataset, 40-scene sample):
#   clear:      mean=0.397  p10=0.135  p50=0.389  p90=0.675
#   fresh_burn: mean=0.187  p10=0.055  p50=0.142  p90=0.398
#   old_burn:   mean=0.296  p10=0.128  p50=0.261  p90=0.525
# -> threshold 0.25 sits just above old_burn's median and below clear's
#    median: catches confusable low-NDVI clear terrain without flagging
#    the majority of ordinary clear land.

In [5]:
LANDMARKS = {"Vesuvius": (14.4260, 40.8210), "Monte Pisano": (10.4800, 43.7300)}

def scene_bbox_wgs84(img_path):
    from rasterio.warp import transform_bounds
    with rasterio.open(img_path) as s:
        return transform_bounds(s.crs, "EPSG:4326", *s.bounds)

landmark_scenes = set()
for scene in phi_scenes:
    try:
        minx, miny, maxx, maxy = scene_bbox_wgs84(f"{PHI_DIR}/images/{scene}_img.tif")
    except Exception:
        continue
    for name, (lon, lat) in LANDMARKS.items():
        if minx <= lon <= maxx and miny <= lat <= maxy:
            landmark_scenes.add(scene)
            print(f"{name} found inside scene {scene}")

if not landmark_scenes:
    print("Neither landmark falls inside any acquired PhiSat-2 scene "
         "- NDVI-based hard-negative mining covers this case generally instead.")

Vesuvius found inside scene effis_278540_2025-08-17T10-09-37
Monte Pisano found inside scene effis_457509_2026-05-22T10-18-38


In [6]:
shutil.rmtree(OUT_ROOT, ignore_errors=True)   # clear any stale dry-run/crash residue first

DRY_RUN_N = 3
dry_rows = []
for scene in phi_scenes[:DRY_RUN_N]:
    split = phi_map[scene]
    with rasterio.open(f"{PHI_DIR}/images/{scene}_img.tif") as s: img, transform, crs = s.read(), s.transform, s.crs
    with rasterio.open(f"{PHI_DIR}/masks/{scene}_mask.tif") as s: msk = s.read(1)
    dry_rows += curate_and_tile(img, msk, scene, "phisat2_sim", "dryrun_" + split, "native",
                                QUANTIFICATION["phisat2_sim"])

n_tiles = len(dry_rows)
print(f"scenes attempted: {DRY_RUN_N} | tiles written: {n_tiles}")
if n_tiles == 0:
    print("!! zero tiles - check max_nodata thresholds and scene dimensions before proceeding")
else:
    n_hard = sum(1 for r in dry_rows if r["hard_negative"])
    n_clear_kept = sum(1 for r in dry_rows if not r["informative"])
    bytes_written = sum(os.path.getsize(f"{OUT_ROOT}/dryrun_{r['split'].replace('dryrun_','')}/{r['tile']}_img.npy") +
                        os.path.getsize(f"{OUT_ROOT}/dryrun_{r['split'].replace('dryrun_','')}/{r['tile']}_mask.npy")
                        for r in dry_rows)
    mb_per_scene = bytes_written / 1e6 / DRY_RUN_N
    print(f"{bytes_written/1e6:.0f} MB written | measured {mb_per_scene:.1f} MB/scene "
         f"(vs 240 MB/scene uncurated -> {mb_per_scene/240*100:.0f}% retained)")
    if n_clear_kept:
        print(f"hard-negative clear tiles: {n_hard}/{n_clear_kept} ({n_hard/n_clear_kept*100:.0f}% of kept clear tiles)")

    retained_frac = mb_per_scene / 240
    proj_phi = len(phi_scenes) * 240 * retained_frac
    proj_heo = len(heo_scenes) * 434 * retained_frac
    proj_multi_gsd = 0
    if ENABLE_MULTI_GSD_TEST:
        n_test = sum(1 for v in phi_map.values() if v == "test")
        proj_multi_gsd = n_test * 240 * 2.836 * retained_frac
    total_proj_gb = (proj_phi + proj_heo + proj_multi_gsd) / 1024
    print(f"\nPROJECTED total: {total_proj_gb:.1f} GB (free: {free_gb():.1f} GB)")
    if total_proj_gb > free_gb() - MIN_FREE_GB:
        print("!! PROJECTED SIZE EXCEEDS SAFE FREE SPACE - reduce keep ratios, "
             "disable ENABLE_MULTI_GSD_TEST, or free more disk before continuing")
    else:
        print("OK to proceed to full run.")

shutil.rmtree(OUT_ROOT, ignore_errors=True)   # clear dry-run tiles before the real run

scenes attempted: 3 | tiles written: 327
322 MB written | measured 107.2 MB/scene (vs 240 MB/scene uncurated -> 45% retained)
hard-negative clear tiles: 45/96 (47% of kept clear tiles)

PROJECTED total: 55.4 GB (free: 117.2 GB)
OK to proceed to full run.


In [7]:
assert free_gb() - MIN_FREE_GB > 0, "Not enough free disk to safely proceed"

all_rows = []
stop = False

for i, scene in enumerate(phi_scenes):
    if stop: break
    if free_gb() < MIN_FREE_GB:
        log.error("Free disk below %d GB - stopping", MIN_FREE_GB); stop = True; break
    split = phi_map[scene]
    with rasterio.open(f"{PHI_DIR}/images/{scene}_img.tif") as s: img, transform, crs = s.read(), s.transform, s.crs
    with rasterio.open(f"{PHI_DIR}/masks/{scene}_mask.tif") as s: msk = s.read(1)

    gsds = TEST_GSDS if (split == "test" and ENABLE_MULTI_GSD_TEST) else {"native": TEST_GSDS["native"]}
    for gsd_tag, target_res in gsds.items():
        ri, rm, _ = resample_scene(img, msk, TEST_GSDS["native"], target_res, transform, crs)
        all_rows += curate_and_tile(ri, rm, scene, "phisat2_sim", split, gsd_tag,
                                    scale=QUANTIFICATION["phisat2_sim"])

    if (i + 1) % 25 == 0:
        log.info("PHI progress: %d/%d scenes, %d tiles, %.1f GB free",
                 i + 1, len(phi_scenes), len(all_rows), free_gb())

for i, scene in enumerate(heo_scenes):
    if stop: break
    if free_gb() < MIN_FREE_GB:
        log.error("Free disk below %d GB - stopping", MIN_FREE_GB); break
    split = heo_map[scene]
    with rasterio.open(f"{HEO_DIR}/images/{scene}_img.tif") as s: img, crs, transform = s.read(), s.crs, s.transform
    with rasterio.open(f"{HEO_DIR}/masks/{scene}_mask.tif") as s: msk = s.read(1)
    all_rows += curate_and_tile(img, msk, scene, "heo", split, "native", scale=QUANTIFICATION["heo"])
    log.info("HEO progress: %d/%d scenes, %d tiles total, %.1f GB free",
             i + 1, len(heo_scenes), len(all_rows), free_gb())

idx = pd.DataFrame(all_rows)
idx.to_csv(f"{OUT_ROOT}/tiles_index.csv", index=False)
print(idx.groupby(["split", "source", "gsd"]).size())
print(f"\ninformative: {idx.informative.sum()}/{len(idx)} ({idx.informative.mean()*100:.1f}%)")
print(f"hard negatives: {idx.hard_negative.sum()} of {(~idx.informative).sum()} clear tiles")
print(f"free disk remaining: {free_gb():.1f} GB")

2026-08-05 23:14:35,217 INFO PHI progress: 25/383 scenes, 5876 tiles, 111.7 GB free
2026-08-05 23:17:57,237 INFO PHI progress: 50/383 scenes, 11595 tiles, 106.0 GB free
2026-08-05 23:19:39,573 INFO PHI progress: 75/383 scenes, 15951 tiles, 101.7 GB free
2026-08-05 23:21:57,460 INFO PHI progress: 100/383 scenes, 21569 tiles, 96.1 GB free
2026-08-05 23:25:13,212 INFO PHI progress: 125/383 scenes, 27756 tiles, 90.0 GB free
2026-08-05 23:28:23,845 INFO PHI progress: 150/383 scenes, 33839 tiles, 84.0 GB free
2026-08-05 23:30:40,670 INFO PHI progress: 175/383 scenes, 39470 tiles, 78.4 GB free
2026-08-05 23:34:40,400 INFO PHI progress: 200/383 scenes, 47615 tiles, 70.3 GB free
2026-08-05 23:36:20,958 INFO PHI progress: 225/383 scenes, 52004 tiles, 65.9 GB free
2026-08-05 23:38:27,711 INFO PHI progress: 250/383 scenes, 55747 tiles, 62.2 GB free
2026-08-05 23:40:40,116 INFO PHI progress: 275/383 scenes, 60985 tiles, 57.0 GB free
2026-08-05 23:43:36,864 INFO PHI progress: 300/383 scenes, 66983 t

split      source       gsd     
heo_test   heo          native        794
heo_train  heo          native       3109
heo_val    heo          native        304
test       phisat2_sim  heo_like    19581
                        native       5865
                        s2_like      1270
train      phisat2_sim  native      50822
val        phisat2_sim  native       6367
dtype: int64

informative: 54414/88112 (61.8%)
hard negatives: 28526 of 33698 clear tiles
free disk remaining: 30.1 GB


In [8]:
os.system(f"du -sh {OUT_ROOT}")
card = {
    "version": "v1", "tile_size": TILE, "bands": BAND_NAMES,
    "band_order": "B,G,R,RE1,RE2,RE3,NIR (PAN excluded)",
    "dtype": "uint16 raw DN; divide by tile's 'scale' column for reflectance",
    "classes": {0:"clear",2:"fresh_burn",3:"old_burn",4:"cloud",5:"cloud_shadow",6:"water",255:"nodata"},
    "priority": "water < old < fresh < shadow < cloud < nodata",
    "curation": {
        "max_nodata_informative": MAX_NODATA_INFORMATIVE,
        "max_nodata_clear": MAX_NODATA_CLEAR,
        "ndvi_hard_negative_threshold": NDVI_HARD_NEG_THRESHOLD,
        "clear_keep_ratio_easy": CLEAR_KEEP_RATIO_EASY,
        "clear_keep_ratio_hard": CLEAR_KEEP_RATIO_HARD,
        "informative_rule": "any fresh/old burn px, or >=5% cloud/shadow/water",
        "hard_negative_rule": f"mean NDVI over clear px < {NDVI_HARD_NEG_THRESHOLD}",
    },
    "heo_excluded_scenes": sorted(HEO_EXCLUDE),
    "test_gsds_m": TEST_GSDS if ENABLE_MULTI_GSD_TEST else {"native": TEST_GSDS["native"]},
    "splits": idx.groupby("split").size().to_dict(),
    "sources": {str(k): v for k, v in idx.groupby(["split","source"]).size().to_dict().items()},
}
json.dump(card, open(f"{OUT_ROOT}/dataset_card.json", "w"), indent=2, default=str)
print(json.dumps(card, indent=2, default=str))

82G	processed/dataset_v1
{
  "version": "v1",
  "tile_size": 256,
  "bands": [
    "B",
    "G",
    "R",
    "RE1",
    "RE2",
    "RE3",
    "NIR"
  ],
  "band_order": "B,G,R,RE1,RE2,RE3,NIR (PAN excluded)",
  "dtype": "uint16 raw DN; divide by tile's 'scale' column for reflectance",
  "classes": {
    "0": "clear",
    "2": "fresh_burn",
    "3": "old_burn",
    "4": "cloud",
    "5": "cloud_shadow",
    "6": "water",
    "255": "nodata"
  },
  "priority": "water < old < fresh < shadow < cloud < nodata",
  "curation": {
    "max_nodata_informative": 0.5,
    "max_nodata_clear": 0.05,
    "ndvi_hard_negative_threshold": 0.25,
    "clear_keep_ratio_easy": 0.1,
    "clear_keep_ratio_hard": 1.0,
    "informative_rule": "any fresh/old burn px, or >=5% cloud/shadow/water",
    "hard_negative_rule": "mean NDVI over clear px < 0.25"
  },
  "heo_excluded_scenes": [
    "IMH04_1CST__OPT8_20260317T123240_20260317T123242_20260323T105659_03985______O_A02",
    "IMH04_1CST__OPT8_20260317T123336_2

In [9]:
import glob, os
import numpy as np
import pandas as pd

CLASS_NAMES = {0: "clear", 2: "fresh_burn", 3: "old_burn", 4: "cloud",
              5: "cloud_shadow", 6: "water", 255: "nodata"}

idx = pd.read_csv(f"{OUT_ROOT}/tiles_index.csv")

# --- Pixel-level balance: mean class fraction across all tiles in each split ---
frac_cols = [c for c in idx.columns if c.startswith("frac_")]
pixel_balance = idx.groupby("split")[frac_cols].mean() * 100
pixel_balance.columns = [c.replace("frac_", "") for c in pixel_balance.columns]
# frac_ columns don't include nodata explicitly (curation tracked it separately);
# derive nodata's average share from nodata_frac
pixel_balance["nodata"] = idx.groupby("split")["nodata_frac"].mean() * 100
pixel_balance = pixel_balance.round(2)
pixel_balance["n_tiles"] = idx.groupby("split").size()
print("=== Mean pixel-class % per tile, by split ===")
print(pixel_balance.to_string())

# --- Tile-level presence: % of tiles in each split containing ANY pixels of each class ---
presence = pd.DataFrame({
    col.replace("frac_", ""): idx.groupby("split")[col].apply(lambda s: (s > 0).mean() * 100)
    for col in frac_cols
})
presence["nodata"] = idx.groupby("split")["nodata_frac"].apply(lambda s: (s > 0).mean() * 100)
presence = presence.round(1)
print("\n=== % of TILES containing at least one pixel of each class, by split ===")
print(presence.to_string())

# --- source breakdown per split, for context ---
print("\n=== source composition per split ===")
print(idx.groupby(["split", "source"]).size().unstack(fill_value=0))

=== Mean pixel-class % per tile, by split ===
               0      2     3      4      5      6  nodata  n_tiles
split                                                              
heo_test    8.19   0.00  0.00  88.28   1.73   0.39    1.41      794
heo_train  58.36   0.98  5.27  30.70   2.22   1.22    1.25     3109
heo_val    66.99   0.00  1.03  17.12  13.21   0.00    1.65      304
test       63.02  18.36  6.78   0.54   0.16  11.10    0.02    26716
train      66.12  15.45  6.50   0.41   0.13  11.35    0.03    50822
val        62.54  17.59  8.84   0.20   0.06  10.76    0.02     6367

=== % of TILES containing at least one pixel of each class, by split ===
              0     2     3     4     5     6  nodata
split                                                
heo_test   19.4   0.0   0.1  96.1  26.7   6.0     6.5
heo_train  79.2   3.1  14.9  43.3  16.4   9.9     6.9
heo_val    95.1   0.0   7.9  45.7  44.4   3.6     9.9
test       79.4  27.4  18.2   1.6   0.9  26.4     0.1
train      8